In [1]:
!pip install torch==2.0.1

!pip install numpy==1.26.4

!pip install transformers==4.40.0

!pip install torchcrf==1.1.0

!pip install spacy==3.7.4
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.9/619.9 MB 2.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 MB 5.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 83.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 57.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.1/557.1 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 9.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 MB 14.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.6/102.6 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.2/173.2 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 MB 9.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━

In [2]:
from __future__ import annotations
import json
from dataclasses import dataclass
from itertools import groupby
from pathlib import Path
from typing import Dict, Iterable, Iterator, List, Optional, Tuple,Sequence,Any
import csv
import os
import random
import string
import sys
from collections import Counter, defaultdict
from pathlib import Path
import torch
from transformers import AutoModel, AutoTokenizer, logging, AutoConfig
import spacy
from transformers import set_seed as hf_set_seed
import inspect

from pathlib import Path
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from transformers import RobertaTokenizerFast


import torch.nn.functional as F
from torch import nn
from transformers import AutoModel
from TorchCRF import CRF

from torch.optim import SGD,AdamW

from transformers import get_linear_schedule_with_warmup

import math
from itertools import groupby



# **Helper function**

In [3]:
@dataclass(frozen=True)
class TokenRecord:
    article_id: str
    chunk_index: int
    sentence_id: int
    span: str
    token: str
    label: int


# Numeric label mapping: 0 -> Outside, 1 -> Begin, 2 -> Inside
LABEL_OUTSIDE = 0
LABEL_BEGIN = 1
LABEL_INSIDE = 2


def resolve_article_path(article_id: str, articles_dir: Path) -> Path:
    file_name = article_id
    if not file_name.startswith("article"):
        file_name = f"article{file_name}"
    path = articles_dir / f"{file_name}.txt"
    if not path.exists():
        raise FileNotFoundError(f"Article file not found: {path}")
    return path


def load_article_text(article_path: Path) -> str:
    return article_path.read_text(encoding="utf8")


def read_token_rows(path: Path) -> Iterator[TokenRecord]:
    """Stream token rows from a tokenizer TSV output."""
    with path.open("r", encoding="utf8") as handle:
        for line_num, raw_line in enumerate(handle, 1):
            parts = raw_line.rstrip("\n").split("\t")
            sentence_id_str = "-1"  # Default to -1 if not present

            if len(parts) == 5:
                article_id, chunk_idx_str, span, token, label_str = parts
            elif len(parts) == 6:
                article_id, chunk_idx_str, sentence_id_str, span, token, label_str = parts
            else:
                continue

            try:
                chunk_index = int(chunk_idx_str)
                sentence_id = int(sentence_id_str)
                label_id = int(label_str)
            except ValueError:
                continue

            yield TokenRecord(
                article_id=article_id,
                chunk_index=chunk_index,
                sentence_id=sentence_id,
                span=span,
                token=token,
                label=label_id,
            )


def group_token_records(records: Iterable[TokenRecord]) -> Iterator[List[TokenRecord]]:
    """Group token records by article and chunk index."""
    keyfunc = lambda rec: (rec.article_id, rec.chunk_index)
    for _, group in groupby(sorted(records, key=keyfunc), key=keyfunc):
        grouped = list(group)
        if grouped:
            yield grouped


def parse_span(span: str) -> Optional[Tuple[int, int]]:
    """Convert a ``start:end`` span string into integer offsets."""
    if not span:
        return None
    try:
        start_str, end_str = span.split(":", 1)
        return int(start_str), int(end_str)
    except (ValueError, AttributeError):
        return None


def load_vocab_mapping(path: Path) -> Dict[str, int]:
    """Load a token-to-id mapping from a JSON vocabulary file."""
    with path.open("r", encoding="utf8") as handle:
        data = json.load(handle)
    return {str(token): int(token_id) for token, token_id in data.items()}


def _adjust_first_label_in_chunk(labels_chunk: List[str]):
    first_label_idx = -1
    for i, label in enumerate(labels_chunk):
        if label.strip() != str(LABEL_OUTSIDE):
            first_label_idx = i
            break
    if first_label_idx != -1 and labels_chunk[first_label_idx].strip() == str(LABEL_INSIDE):
        labels_chunk[first_label_idx] = str(LABEL_BEGIN)


def read_tokens(
    path: Path,
    *,
    delimiter: str = "\t",
    token_column: int = 3,
) -> Iterable[str]:
    """Stream tokens from a tokenized TSV file."""
    csv.field_size_limit(sys.maxsize)
    with path.open("r", encoding="utf8", newline="") as handle:
        reader = csv.reader(handle, delimiter=delimiter)
        for row in reader:
            if len(row) <= token_column:
                continue
            token = row[token_column]
            if not token:
                continue
            yield token


def should_keep_token(token: str) -> bool:
    """Return True if the token should remain in the vocabulary output."""
    stripped_token = token.lstrip("ĠĊ")
    if not stripped_token:
        return True

    if (
        (len(set(stripped_token)) == 1 and stripped_token[0] in string.punctuation and len(stripped_token) > 1)
        or (not stripped_token.isascii() or any(ch not in string.printable for ch in stripped_token))
        or (sum(ch in string.punctuation or ch.isdigit() for ch in stripped_token) / len(stripped_token) >= 0.66)
        or (stripped_token in {"),;", "?!", "?!?", "!?", "??", "..)"})
    ):
        return False

    return True


def build_vocab(
    tokens: Iterable[str],
    *,
    tokenizer,
    special_tokens: Iterable[str] = (),
    min_frequency: int = 1,
    max_size: Optional[int] = None,
) -> List[Tuple[str, int]]:
    """Select tokens present in a pretrained tokenizer and map them to IDs."""
    frequency = Counter(tokens)
    vocab: List[Tuple[str, int]] = []
    seen: set[str] = set()

    def convert(token: str) -> int:
        token_id = tokenizer.convert_tokens_to_ids(token)
        return int(token_id)

    for token in special_tokens:
        if token in seen:
            continue
        vocab.append((token, convert(token)))
        frequency.pop(token, None)
        seen.add(token)

    sorted_candidates = sorted(
        (
            (token, count)
            for token, count in frequency.items()
            if count >= max(1, min_frequency) and token not in seen and should_keep_token(token)
        ),
        key=lambda item: (-item[1], item[0]),
    )

    unk_id = getattr(tokenizer, "unk_token_id", None)

    for token, _ in sorted_candidates:
        token_id = convert(token)
        if unk_id is not None and token_id == unk_id:
            continue
        vocab.append((token, token_id))
        seen.add(token)
        if max_size is not None and len(vocab) >= max_size:
            break

    return vocab


def write_vocab(entries: List[Tuple[str, int]], path: Path) -> None:
    """Persist the vocabulary as a JSON token-to-id mapping."""
    mapping = {token: token_id for token, token_id in entries}
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf8") as handle:
        json.dump(mapping, handle, ensure_ascii=False, indent=2, sort_keys=True)


def _collect_spans(sequence: Sequence[int]) -> List[Tuple[int, int]]:
    spans: List[Tuple[int, int]] = []
    start: Optional[int] = None
    for idx, label in enumerate(sequence):
        if label == LABEL_BEGIN:
            if start is not None and start < idx:
                spans.append((start, idx))
            start = idx
        elif label == LABEL_INSIDE:
            pass
        else:
            if start is not None and start < idx:
                spans.append((start, idx))
            start = None
    if start is not None and start < len(sequence):
        spans.append((start, len(sequence)))
    return spans


def _get_span_texts(spans: List[Tuple[int, int]], tokens: List[str], tokenizer) -> List[List[str]]:
    span_texts = []
    for start, end in spans:
        span_tokens = tokens[start:end]
        span_texts.append(span_tokens)
    return span_texts


def _span_overlap(span_a: Tuple[int, int], span_b: Tuple[int, int]) -> int:
    start = max(span_a[0], span_b[0])
    end = min(span_a[1], span_b[1])
    return max(0, end - start)

def load_tokenizer(model_name: str):
    """Load a tokenizer configured for byte-level BPE with prefix spacing."""
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if hasattr(tokenizer, "add_prefix_space"):
        tokenizer.add_prefix_space = True
    return tokenizer



# **Labeling raw data**

In [4]:

def tokenize_with_offsets(tokenizer: AutoTokenizer, text: str) -> Tuple[List[str], List[Tuple[int, int]]]:
    """Tokenize full article text to get tokens and character offsets without triggering
    model max-length warnings. We chunk manually later, so we temporarily raise
    the tokenizer.model_max_length to avoid the standard HF warning
    (e.g., "704 > 512").

    This does NOT truncate; chunking is handled by chunk_tokens_with_stride.
    """
    old_max_len = getattr(tokenizer, "model_max_length", None)
    try:
        # If model_max_length is small (e.g., 512), temporarily raise it to a large value
        # so HF doesn't warn when we tokenize long documents.
        if isinstance(old_max_len, int) and old_max_len <= 2048:
            tokenizer.model_max_length = 1_000_000  # effectively "no limit" for our purposes
        encoding = tokenizer(
            text,
            return_offsets_mapping=True,
            add_special_tokens=False,
            truncation=False,  # explicit: we do not want truncation here
        )
    finally:
        if old_max_len is not None:
            tokenizer.model_max_length = old_max_len
    return encoding.tokens(), list(encoding["offset_mapping"])


def label_token(start: int, end: int, spans: Iterable[Tuple[int, int]]) -> str:
    candidates = [(s, e) for (s, e) in spans if start >= s and end <= e]
    if not candidates:
        return str(LABEL_OUTSIDE)
    span_start, span_end = min(candidates, key=lambda x: x[1] - x[0])
    if start == span_start:
        return str(LABEL_BEGIN)
    return str(LABEL_INSIDE)


def chunk_tokens_with_stride(
    tokens: List[str],
    offsets: List[Tuple[int, int]],
    labels: List[str],
    sentence_ids: List[int],
    max_tokens: int,
    stride: int
) -> List[Tuple[List[str], List[Tuple[int, int]], List[str], List[int]]]:
    chunks: List[Tuple[List[str], List[Tuple[int, int]], List[str], List[int]]] = []
    start_idx = 0
    while start_idx < len(tokens):
        end_idx = min(start_idx + max_tokens, len(tokens))
        chunks.append((
            tokens[start_idx:end_idx],
            offsets[start_idx:end_idx],
            labels[start_idx:end_idx],
            sentence_ids[start_idx:end_idx]
        ))
        if end_idx == len(tokens):
            break
        start_idx += max_tokens - stride
    return chunks




def create_token_labels(
    *,
    articles_dir: Path,
    tokens_output: Path,
    tokenizer: AutoTokenizer,
    model: AutoModel,
    device: torch.device,
    max_tokens: int,
    stride: int,
    nlp=None,
    spans_file: Path | None = None,
) -> None:
    # Lazy-load spaCy model if not provided
    if nlp is None:
        try:
            nlp = spacy.load("en_core_web_sm")
        except Exception:
            # Fallback to a simple rule-based sentencizer if the model isn't available
            nlp = spacy.blank("en")
            if "sentencizer" not in nlp.pipe_names:
                nlp.add_pipe("sentencizer")

    span_map = load_spans(spans_file) if spans_file else defaultdict(list)

    tokens_output.parent.mkdir(parents=True, exist_ok=True)
    with tokens_output.open("w", encoding="utf8") as tokens_f:
        article_files = sorted([p for p in articles_dir.glob("*.txt") if p.is_file()])
        for article_path in article_files:
            article_id = article_path.stem.replace("article", "")
            spans = span_map.get(article_id, [])

            try:
                article_text = load_article_text(article_path)
            except FileNotFoundError as err:
                print(f"Skipping {article_id}: {err}", file=sys.stderr)
                continue

            # Perform sentence segmentation
            doc = nlp(article_text)
            sentence_map = {}
            for i, sent in enumerate(doc.sents):
                for char_idx in range(sent.start_char, sent.end_char):
                    sentence_map[char_idx] = i

            tokens, offsets = tokenize_with_offsets(tokenizer, article_text)
            token_labels = [label_token(start, end, spans) for (start, end) in offsets]
            token_sentence_ids = [sentence_map.get(start, -1) for start, end in offsets]

            for chunk_idx, (tokens_chunk, offsets_chunk, labels_chunk, sentence_ids_chunk) in enumerate(
                chunk_tokens_with_stride(tokens, offsets, token_labels, token_sentence_ids, max_tokens, stride)
            ):
                _adjust_first_label_in_chunk(labels_chunk)

                for (start, end), token, label, sentence_id in zip(offsets_chunk, tokens_chunk, labels_chunk, sentence_ids_chunk):
                    tokens_f.write(f"{article_id}\t{chunk_idx}\t{sentence_id}\t{start}:{end}\t{token}\t{label}\n")


def load_spans(labels_path: Path) -> Dict[str, List[Tuple[int, int]]]:
    """Load gold spans from a Task SI labels file.

    Expected format per line (TAB-separated):
        article_id    start_offset    end_offset

    Returns a mapping from article_id (str) -> list of (start, end) tuples.
    """
    spans: Dict[str, List[Tuple[int, int]]] = defaultdict(list)
    if labels_path is None or not labels_path.exists():
        return spans

    with labels_path.open("r", encoding="utf8") as handle:
        for line in handle:
            parts = line.strip().split("\t")
            if len(parts) < 3:
                continue
            article_id, start_str, end_str = parts[0], parts[1], parts[2]
            try:
                start, end = int(start_str), int(end_str)
            except ValueError:
                continue
            spans[article_id].append((start, end))

    return spans


# **Create data loader**

In [5]:


class PropagandaTokenDataset(Dataset):
    def __init__(
        self,
        chunks: Sequence[List[TokenRecord]],
        tokenizer,
        max_length: int = 256,
        vocab_map: Optional[Dict[str, int]] = None,
    ) -> None:
        self._chunks = list(chunks)
        self._tokenizer = tokenizer
        self._max_length = max_length
        self._vocab_map = dict(vocab_map or {})

    def __len__(self) -> int:
        return len(self._chunks)

    def __getitem__(self, index: int):
        chunk = self._chunks[index]
        tokens = [record.token for record in chunk]
        labels = [record.label for record in chunk]
        sentence_ids = [record.sentence_id for record in chunk]

        cls_id = getattr(self._tokenizer, "cls_token_id", None)
        sep_id = getattr(self._tokenizer, "sep_token_id", None)
        pad_id = getattr(self._tokenizer, "pad_token_id", 0)
        unk_id = getattr(self._tokenizer, "unk_token_id", None)
        unk_token = getattr(self._tokenizer, "unk_token", None)
        if unk_id is None and unk_token is not None:
            unk_id = self._tokenizer.convert_tokens_to_ids(unk_token)
        if unk_id is None:
            unk_id = 0

        available_length = self._max_length
        special_count = 0
        if cls_id is not None:
            special_count += 1
        if sep_id is not None:
            special_count += 1
        available_length = max(0, available_length - special_count)

        tokens = tokens[:available_length]
        labels = labels[:available_length]
        sentence_ids = sentence_ids[:available_length]

        token_ids = self._convert_tokens_to_ids(tokens, unk_id)

        input_ids = []
        label_ids = []
        attention_mask = []
        sentence_ids_padded = []

        self._add_special_tokens(cls_id, input_ids, label_ids, attention_mask, sentence_ids_padded)

        input_ids.extend(token_ids)
        label_ids.extend(labels)
        attention_mask.extend([1] * len(token_ids))
        sentence_ids_padded.extend(sentence_ids)

        self._add_special_tokens(sep_id, input_ids, label_ids, attention_mask, sentence_ids_padded)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(label_ids, dtype=torch.long),
            "sentence_ids": torch.tensor(sentence_ids_padded, dtype=torch.long),
            "pad_token_id": pad_id,
        }

    def _convert_tokens_to_ids(self, tokens: List[str], unk_id: int) -> List[int]:
        token_ids = []
        for token in tokens:
            if token in self._vocab_map:
                token_id = self._vocab_map[token]
            else:
                token_id = self._tokenizer.convert_tokens_to_ids(token)
            if token_id is None:
                token_id = unk_id
            token_ids.append(token_id)
        return token_ids

    def _add_special_tokens(self, special_id: Optional[int], input_ids: List[int], label_ids: List[int], attention_mask: List[int], sentence_ids_padded: List[int]):
        if special_id is not None:
            input_ids.append(special_id)
            label_ids.append(-100)
            attention_mask.append(1)
            sentence_ids_padded.append(-100)

def build_collate_fn(pad_token_id: int):
    def _collate(batch):
        input_ids = pad_sequence(
            [item["input_ids"] for item in batch], batch_first=True, padding_value=pad_token_id
        )
        attention_mask = pad_sequence(
            [item["attention_mask"] for item in batch], batch_first=True, padding_value=0
        )
        labels = pad_sequence(
            [item["labels"] for item in batch], batch_first=True, padding_value=-100
        )
        sentence_ids = pad_sequence(
            [item["sentence_ids"] for item in batch], batch_first=True, padding_value=-100
        )
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "sentence_ids": sentence_ids,
        }

    return _collate

def create_data_loader(
    data: Path,
    tokenizer,
    *,
    vocab_map: Optional[Dict[str, int]] = None,
    batch_size: int = 8,
    shuffle: bool = True,
    max_length: int = 256,
) -> DataLoader:
    """
    Create a DataLoader from either:
      - a TSV file path containing token data, or
      - preprocessed chunks of TokenRecord lists.
    """

    rows = read_token_rows(data)
    chunks = list(group_token_records(rows))


    dataset = PropagandaTokenDataset(chunks, tokenizer, max_length=max_length, vocab_map=vocab_map)
    pad_token_id = getattr(tokenizer, "pad_token_id", 0)
    collate_fn = build_collate_fn(pad_token_id)

    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_fn)

# **Training model**

Roberta baseline

In [6]:
class RobertaLargeBaseline(nn.Module):
    """
    Baseline model for token classification:
    - Roberta-large encoder
    - Linear classification layer
    - Optional dropout
    """
    def __init__(
        self,
        model_name: str = "roberta-large",
        num_labels: int = 3,  # O, B, I
        dropout: float = 0.1,
        label_smoothing: float = 0.0,
        class_weights: Optional[Tuple[float, ...]] = None,
    ):
        super().__init__()
        config = AutoConfig.from_pretrained(model_name, num_labels=num_labels)
        self.roberta = AutoModel.from_pretrained(model_name, config=config)
        hidden_size = config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)
        self.num_labels = num_labels
        self.label_smoothing = label_smoothing

        if class_weights is not None:
            if len(class_weights) != num_labels:
                raise ValueError("class_weights must match num_labels")
            self.register_buffer("class_weights", torch.tensor(class_weights, dtype=torch.float))
        else:
            self.class_weights = None

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
    ):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = self.dropout(outputs.last_hidden_state)
        logits = self.classifier(hidden_states)

        result = {"logits": logits}

        if labels is not None:
            loss = F.cross_entropy(
                logits.view(-1, self.num_labels),
                labels.view(-1),
                ignore_index=-100,
                reduction="mean",
                label_smoothing=self.label_smoothing,
                weight=self.class_weights,
            )
            result["loss"] = loss

        predictions = torch.argmax(logits, dim=-1)
        result["predictions"] = predictions
        return result

CRF

In [7]:
class TokenCRFModel(nn.Module):
    """RoBERTa encoder followed by a CRF for BIO tagging."""

    def __init__(
        self,
        model_name: str,
        num_labels: int,
        dropout: float = 0.1,
        aux_ce_weight: float = 0.0,
        label_smoothing: float = 0.0,
        class_weights: Optional[Tuple[float, ...]] = None,
    ) -> None:
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)
        self.crf = CRF(num_labels, pad_idx=None, use_gpu=torch.cuda.is_available())
        self.num_labels = num_labels
        self.aux_ce_weight = aux_ce_weight
        self.label_smoothing = label_smoothing
        if class_weights is not None:
            if len(class_weights) != num_labels:
                raise ValueError("class_weights must match num_labels")
            self.register_buffer(
                "class_weights",
                torch.tensor(class_weights, dtype=torch.float),
            )
        else:
            self.class_weights = None

        # Add BIO constraints to the CRF layer to prevent malformed sequences.
        # 0: O, 1: B, 2: I
        o_id, b_id, i_id = 0, 1, 2
        self.crf.start_trans.data[i_id] = -10000.0      # Can't start with I
        self.crf.trans_matrix.data[o_id, i_id] = -10000.0  # O -> I is forbidden
        self.crf.trans_matrix.data[b_id, b_id] = -10000.0  # B -> B is forbidden
        self.crf.trans_matrix.data[i_id, b_id] = -10000.0  # I -> B is forbidden

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
        **kwargs, # to ignore other arguments
    ) -> Dict[str, torch.Tensor]:
        """Run the encoder and CRF, returning loss, predictions, and confidence scores."""

        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs.last_hidden_state)
        emissions = self.classifier(sequence_output)
        mask = attention_mask.bool()
        result: Dict[str, torch.Tensor] = {}

        if labels is not None:
            label_mask = labels != -100
            effective_mask = mask & label_mask
            crf_labels = labels.masked_fill(~label_mask, 0)
            log_likelihood = self.crf(emissions, crf_labels, mask=effective_mask)
            token_counts = effective_mask.float().sum(dim=1).clamp(min=1.0)
            crf_loss = (-log_likelihood / token_counts).mean()
            loss = crf_loss
            if self.aux_ce_weight > 0:
                ce_loss = F.cross_entropy(
                    emissions.view(-1, self.num_labels),
                    labels.view(-1),
                    ignore_index=-100,
                    reduction="mean",
                    label_smoothing=self.label_smoothing,
                    weight=self.class_weights,
                )
                loss = (1 - self.aux_ce_weight) * crf_loss + self.aux_ce_weight * ce_loss
            result["loss"] = loss
            decode_mask = effective_mask
        else:
            decode_mask = mask

        predictions = self.crf.viterbi_decode(emissions, mask=decode_mask)
        
        # Also return confidence scores (average log-probability of the predicted sequence)
        log_probs = F.log_softmax(emissions, dim=-1)
        confidence_scores = []
        for i, seq in enumerate(predictions):
            seq_len = int(decode_mask[i].sum())
            if seq_len == 0:
                confidence_scores.append(torch.tensor(0.0, device=emissions.device))
                continue
            
            # Gather the log-probabilities of the predicted tags for the actual sequence length
            seq_log_probs = log_probs[i, :seq_len].gather(1, torch.tensor(seq[:seq_len], device=log_probs.device).unsqueeze(1)).squeeze(1)
            
            # Calculate the average log-probability
            avg_log_prob = seq_log_probs.mean()
            confidence_scores.append(avg_log_prob)

        result["predictions"] = predictions
        result["confidence_scores"] = torch.stack(confidence_scores)
        return result

MGN + CRF

In [8]:
def focal_loss(
    logits: torch.Tensor,
    labels: torch.Tensor,
    alpha: float = 0.25,
    gamma: float = 2.0,
    ignore_index: int = -100,
) -> torch.Tensor:
    """
    Focal Loss - automatically focus on hard examples
    """
    num_classes = logits.size(-1)

    # Reshape
    logits_flat = logits.view(-1, num_classes)
    labels_flat = labels.view(-1)

    # Filter ignored labels
    valid_mask = (labels_flat != ignore_index)
    logits_valid = logits_flat[valid_mask]
    labels_valid = labels_flat[valid_mask]

    # Check empty
    if labels_valid.numel() == 0:
        return torch.tensor(0.0, device=logits.device, requires_grad=True)

    # Cross entropy loss
    ce_loss = F.cross_entropy(
        logits_valid,
        labels_valid,
        reduction="none",
    )

    # Compute p_t
    p = F.softmax(logits_valid, dim=-1)
    labels_one_hot = F.one_hot(labels_valid, num_classes=num_classes).float()
    p_t = (p * labels_one_hot).sum(dim=-1)

    # Focal weight
    focal_weight = (1 - p_t) ** gamma

    # Final loss
    loss = (alpha * focal_weight * ce_loss).mean()

    return loss


class MultiGranularityTokenCRFModel(nn.Module):
    """
    Multi-granularity model with:
    - Sentence-level gating
    - Token-level classification
    - CRF layer for structured prediction
    - Scheduled focal loss with warm-up
    """

    def __init__(
        self,
        model_name: str,
        num_labels: int,
        dropout: float = 0.1,
        sentence_loss_weight: float = 0.8,
        label_smoothing: float = 0.0,
        class_weights: Optional[Tuple[float, ...]] = None,
        gate_function: str = "sigmoid",
        apply_gate_in_inference: bool = True,
        # CRF parameters
        use_crf: bool = True,
        crf_loss_weight: float = 0.5,  # Balance between CRF and CE/Focal loss
        enforce_bio_constraints: bool = True,  # Add BIO constraints to CRF
        # Focal loss parameters
        use_focal_loss: bool = True,
        focal_alpha: float = 0.75,
        focal_gamma: float = 3.0,
        # Warm-up parameters
        focal_warmup_steps: int = 3,
        focal_warmup_method: str = "linear",
    ) -> None:
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name, output_hidden_states=True)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)

        self.sentence_classifier = nn.Linear(hidden_size, 1)
        self.token_classifier = nn.Linear(hidden_size, num_labels)

        self.num_labels = num_labels
        self.sentence_loss_weight = sentence_loss_weight
        self.label_smoothing = label_smoothing
        self.gate_function = gate_function
        self.apply_gate_in_inference = apply_gate_in_inference

        # CRF layer
        self.use_crf = use_crf
        self.crf_loss_weight = crf_loss_weight
        if use_crf:
            self.crf = CRF(num_labels, pad_idx=None, use_gpu=torch.cuda.is_available())

            # Add BIO constraints (O=0, B=1, I=2)
            if enforce_bio_constraints and num_labels == 3:
                o_id, b_id, i_id = 0, 1, 2
                self.crf.start_trans.data[i_id] = -10000.0      # Can't start with I
                self.crf.trans_matrix.data[o_id, i_id] = -10000.0  # O -> I is forbidden
                self.crf.trans_matrix.data[b_id, b_id] = -10000.0  # B -> B is forbidden
                self.crf.trans_matrix.data[i_id, b_id] = -10000.0  # I -> B is forbidden

        self.use_focal_loss = use_focal_loss
        self.focal_alpha = focal_alpha
        self.focal_gamma = focal_gamma
        self.focal_warmup_steps = focal_warmup_steps
        self.focal_warmup_method = focal_warmup_method

        self.register_buffer("_training_step", torch.tensor(0))

        if class_weights is not None:
            if len(class_weights) != num_labels:
                raise ValueError("class_weights must match num_labels")
            self.register_buffer(
                "class_weights",
                torch.tensor(class_weights, dtype=torch.float),
            )
        else:
            self.class_weights = None

    def _get_current_focal_gamma(self) -> float:
        """Compute current gamma based on warm-up schedule"""
        if not self.training or not self.use_focal_loss:
            return self.focal_gamma

        current_step = self._training_step.item()

        if self.focal_warmup_method == "step":
            if current_step < self.focal_warmup_steps:
                return 0.0
            else:
                return self.focal_gamma

        elif self.focal_warmup_method == "linear":
            if current_step < self.focal_warmup_steps:
                return self.focal_gamma * (current_step / self.focal_warmup_steps)
            else:
                return self.focal_gamma

        elif self.focal_warmup_method == "cosine":
            if current_step < self.focal_warmup_steps:
                progress = current_step / self.focal_warmup_steps
                return self.focal_gamma * (1 - math.cos(progress * math.pi)) / 2
            else:
                return self.focal_gamma

        return self.focal_gamma

    def _compute_sentence_gates(
        self,
        hidden_states: torch.Tensor,
        sentence_ids: torch.Tensor,
        labels: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Optional[torch.Tensor]]:
        """Compute sentence-level gates"""
        batch_size = hidden_states.size(0)
        seq_len = hidden_states.size(1)

        gates = torch.ones(batch_size, seq_len, device=hidden_states.device)

        sentence_logits_list = []
        sentence_labels_list = []

        for i in range(batch_size):
            sample_hidden_states = hidden_states[i]
            sample_sentence_ids = sentence_ids[i]

            unique_sentence_ids = torch.unique(sample_sentence_ids[sample_sentence_ids != -100])
            if unique_sentence_ids.numel() == 0:
                continue

            # Compute sentence representations
            sentence_representations = []
            for sentence_id in unique_sentence_ids:
                token_indices = (sample_sentence_ids == sentence_id).nonzero(as_tuple=True)[0]
                sentence_representation = sample_hidden_states[token_indices].mean(dim=0)
                sentence_representations.append(sentence_representation)

            sentence_representations = torch.stack(sentence_representations)
            sentence_logits = self.sentence_classifier(sentence_representations).squeeze(-1)

            # Compute gates
            if self.gate_function == "relu":
                sentence_gates = torch.relu(sentence_logits)
            else:
                sentence_gates = torch.sigmoid(sentence_logits)

            # Assign gates to tokens
            for j, sentence_id in enumerate(unique_sentence_ids):
                token_indices = (sample_sentence_ids == sentence_id).nonzero(as_tuple=True)[0]
                gates[i, token_indices] = sentence_gates[j]

            # Collect for loss computation
            if labels is not None:
                sentence_logits_list.append(sentence_logits)

                sample_labels = labels[i]
                sentence_labels = []
                for sentence_id in unique_sentence_ids:
                    token_indices = (sample_sentence_ids == sentence_id).nonzero(as_tuple=True)[0]
                    sentence_label = (sample_labels[token_indices] > 0).any().float()
                    sentence_labels.append(sentence_label)
                sentence_labels = torch.stack(sentence_labels)
                sentence_labels_list.append(sentence_labels)

        sentence_logits_cat = torch.cat(sentence_logits_list) if sentence_logits_list else None
        sentence_labels_cat = torch.cat(sentence_labels_list) if sentence_labels_list else None

        return gates, sentence_logits_cat, sentence_labels_cat

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
        sentence_ids: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        # Increment step counter
        if self.training and labels is not None:
            self._training_step += 1

        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.hidden_states[-1]
        hidden_states = self.dropout(hidden_states)

        result: Dict[str, torch.Tensor] = {}
        emissions = self.token_classifier(hidden_states)  # Token logits/emissions

        # Without sentence-level gating
        if sentence_ids is None:
            if labels is not None:
                # Prepare masks for CRF
                mask = attention_mask.bool()
                label_mask = labels != -100
                effective_mask = mask & label_mask

                # Initialize total loss
                total_loss = 0.0
                loss_components = {}

                # CRF loss
                if self.use_crf:
                    crf_labels = labels.masked_fill(~label_mask, 0)
                    log_likelihood = self.crf(emissions, crf_labels, mask=effective_mask)
                    token_counts = effective_mask.float().sum(dim=1).clamp(min=1.0)
                    crf_loss = (-log_likelihood / token_counts).mean()
                    total_loss += self.crf_loss_weight * crf_loss
                    loss_components["crf_loss"] = crf_loss.item()

                # CE or Focal loss
                current_gamma = self._get_current_focal_gamma()

                if self.use_focal_loss and current_gamma > 0:
                    token_loss = focal_loss(
                        emissions,
                        labels,
                        alpha=self.focal_alpha,
                        gamma=current_gamma,
                        ignore_index=-100,
                    )
                else:
                    token_loss = F.cross_entropy(
                        emissions.view(-1, self.num_labels),
                        labels.view(-1),
                        ignore_index=-100,
                        reduction="mean",
                        weight=self.class_weights,
                    )

                if self.use_crf:
                    total_loss += (1 - self.crf_loss_weight) * token_loss
                else:
                    total_loss = token_loss

                loss_components["token_loss"] = token_loss.item()
                loss_components["current_focal_gamma"] = current_gamma

                result["loss"] = total_loss
                result.update(loss_components)

            # Predictions
            if self.use_crf:
                mask = attention_mask.bool()
                if labels is not None:
                    decode_mask = effective_mask
                else:
                    decode_mask = mask
                predictions = self.crf.viterbi_decode(emissions, mask=decode_mask)
                result["predictions"] = predictions
            else:
                result["predictions"] = torch.argmax(emissions, dim=-1)

            return result

        # With sentence-level gating
        gates, sentence_logits_cat, sentence_labels_cat = self._compute_sentence_gates(
            hidden_states, sentence_ids, labels
        )

        result["gates"] = gates

        # Apply gating
        gates_expanded = gates.unsqueeze(-1)
        emissions_gated = emissions * gates_expanded

        if labels is not None and sentence_logits_cat is not None:
            # Sentence-level loss
            num_positives = sentence_labels_cat.sum()
            num_negatives = len(sentence_labels_cat) - num_positives
            pos_weight = num_negatives / num_positives if num_positives > 0 else torch.tensor(1.0, device=hidden_states.device)

            sentence_loss = F.binary_cross_entropy_with_logits(
                sentence_logits_cat,
                sentence_labels_cat,
                reduction="mean",
                pos_weight=pos_weight
            )

            # Token-level loss with CRF
            mask = attention_mask.bool()
            label_mask = labels != -100
            effective_mask = mask & label_mask

            total_token_loss = 0.0

            # CRF loss on gated emissions
            if self.use_crf:
                crf_labels = labels.masked_fill(~label_mask, 0)
                log_likelihood = self.crf(emissions_gated, crf_labels, mask=effective_mask)
                token_counts = effective_mask.float().sum(dim=1).clamp(min=1.0)
                crf_loss = (-log_likelihood / token_counts).mean()
                total_token_loss += self.crf_loss_weight * crf_loss
                result["crf_loss"] = crf_loss.item()

            # CE or Focal loss
            current_gamma = self._get_current_focal_gamma()

            if self.use_focal_loss and current_gamma > 0:
                token_loss = focal_loss(
                    emissions_gated,
                    labels,
                    alpha=self.focal_alpha,
                    gamma=current_gamma,
                    ignore_index=-100,
                )
            else:
                token_loss = F.cross_entropy(
                    emissions_gated.view(-1, self.num_labels),
                    labels.view(-1),
                    ignore_index=-100,
                    reduction="mean",
                    label_smoothing=self.label_smoothing,
                    weight=self.class_weights,
                )

            if self.use_crf:
                total_token_loss += (1 - self.crf_loss_weight) * token_loss
            else:
                total_token_loss = token_loss

            # Combined loss
            total_loss = self.sentence_loss_weight * sentence_loss + (1 - self.sentence_loss_weight) * total_token_loss

            result["loss"] = total_loss
            result["sentence_loss"] = sentence_loss.item()
            result["token_loss"] = token_loss.item()
            result["current_focal_gamma"] = current_gamma

        # Predictions
        if self.use_crf:
            mask = attention_mask.bool()
            if labels is not None:
                decode_mask = effective_mask
            else:
                decode_mask = mask

            # Use gated or non-gated emissions based on config
            if self.apply_gate_in_inference:
                predictions = self.crf.viterbi_decode(emissions_gated, mask=decode_mask)
            else:
                predictions = self.crf.viterbi_decode(emissions, mask=decode_mask)
            result["predictions"] = predictions
        else:
            if self.apply_gate_in_inference:
                result["predictions"] = torch.argmax(emissions_gated, dim=-1)
            else:
                result["predictions"] = torch.argmax(emissions, dim=-1)

        return result


Optimizer

In [9]:
def prepare_optimizer(
    model: nn.Module,
    learning_rate: float,
    weight_decay: float,
    momentum: float = 0.9,
) -> SGD:
    """Create an SGD optimizer with weight decay."""

    return SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay, momentum=momentum)

def prepare_adamw_optimizer(
    model: nn.Module,
    learning_rate: float,
    weight_decay: float,
) -> AdamW:
    """Create an AdamW optimizer with weight decay."""

    return AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

def create_scheduler(optimizer, total_steps: int, warmup_ratio: float = 0.0):
    """Build a linear warmup/decay scheduler tailored to the total training steps."""

    warmup_steps = int(total_steps * warmup_ratio)
    return get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

# Evaluate function helper

In [10]:
def _span_overlap(span_a: Tuple[int, int], span_b: Tuple[int, int]) -> int:
    start = max(span_a[0], span_b[0])
    end = min(span_a[1], span_b[1])
    return max(0, end - start)


def calculate_span_metrics(
    predictions,
    labels: torch.Tensor,
    mask: torch.Tensor,
) -> Dict[str, float]:
    """Compute cumulative precision/recall numerators and counts for span-level metrics.

    predictions: list[list[int]] from CRF.viterbi_decode or tensor [B, T]
    labels: tensor [B, T]
    mask: tensor [B, T] (True for valid label positions)
    """
    # Normalize predictions to list of lists
    if isinstance(predictions, torch.Tensor):
        preds_list = [pred[: int(m.sum().item())].tolist() for pred, m in zip(predictions, mask)]
    else:
        preds_list = [list(map(int, seq)) for seq in predictions]

    labels_list = [lab[m].tolist() for lab, m in zip(labels, mask)]

    cumulative_prec = 0.0
    cumulative_rec = 0.0
    pred_span_count = 0
    gold_span_count = 0
    pred_malformed = 0
    gold_malformed = 0

    def spans_from_labels(seq: Sequence[int]) -> List[Tuple[int, int]]:
        return _collect_spans(seq)

    def malformed_count(seq: Sequence[int]) -> int:
        # Count tokens labeled I that are not inside a span started by B
        inside = False
        count = 0
        for tag in seq:
            if tag == LABEL_BEGIN:  # B
                inside = True
            elif tag == LABEL_INSIDE:  # I
                if not inside:
                    count += 1
            else:  # O
                inside = False
        return count

    for pred_seq, gold_seq in zip(preds_list, labels_list):
        pred_spans = spans_from_labels(pred_seq)
        gold_spans = spans_from_labels(gold_seq)

        pred_span_count += len(pred_spans)
        gold_span_count += len(gold_spans)
        pred_malformed += malformed_count(pred_seq)
        gold_malformed += malformed_count(gold_seq)

        # Precision-like sum over predicted spans
        for p_start, p_end in pred_spans:
            p_len = max(1, p_end - p_start)
            best_overlap = 0
            for g_start, g_end in gold_spans:
                overlap = _span_overlap((p_start, p_end), (g_start, g_end))
                if overlap > best_overlap:
                    best_overlap = overlap
            cumulative_prec += best_overlap / p_len

        # Recall-like sum over gold spans
        for g_start, g_end in gold_spans:
            g_len = max(1, g_end - g_start)
            best_overlap = 0
            for p_start, p_end in pred_spans:
                overlap = _span_overlap((p_start, p_end), (g_start, g_end))
                if overlap > best_overlap:
                    best_overlap = overlap
            cumulative_rec += best_overlap / g_len

    return {
        "cumulative_prec": cumulative_prec,
        "cumulative_rec": cumulative_rec,
        "pred_span_count": pred_span_count,
        "gold_span_count": gold_span_count,
        "pred_malformed": pred_malformed,
        "gold_malformed": gold_malformed,
    }


def generate_evaluation_debug_report(
    step: int,
    batch_on_device: Dict[str, torch.Tensor],
    outputs: Dict[str, Any],
    labels: torch.Tensor,
    mask: torch.Tensor,
    predictions,
    tokenizer: AutoTokenizer,
) -> List[str]:
    """Generate a human-readable debug report for a batch.

    Includes token text, gold tags, and predicted tags for the first few examples.
    """
    report = [f"Batch {step} debug report:"]
    input_ids = batch_on_device["input_ids"].detach().cpu()
    attention_mask = batch_on_device["attention_mask"].detach().cpu()
    labels_cpu = labels.detach().cpu()
    mask_cpu = mask.detach().cpu()

    if isinstance(predictions, torch.Tensor):
        pred_cpu = predictions.detach().cpu()
        pred_seqs = [pred[: int(m.sum().item())].tolist() for pred, m in zip(pred_cpu, mask_cpu)]
    else:
        pred_seqs = [list(map(int, seq)) for seq in predictions]

    for i in range(input_ids.size(0)):
        seq_len = int(attention_mask[i].sum().item())
        tokens = tokenizer.convert_ids_to_tokens(input_ids[i, :seq_len].tolist())
        gold_seq = labels_cpu[i, :seq_len].tolist()
        gold_seq = [x if x != -100 else 0 for x in gold_seq]
        pred_seq = pred_seqs[i][:seq_len]
        report.append("TOK:\t" + " ".join(tokens))
        report.append("GOLD:\t" + " ".join(map(str, gold_seq)))
        report.append("PRED:\t" + " ".join(map(str, pred_seq)))
        report.append("")
    return report


# **Train function**

In [11]:

@dataclass
class TrainConfig:
    # Paths are no longer required since loaders are provided externally.
    tokens_path: Optional[Path] = None
    vocab_path: Optional[Path] = None
    model_name: str ="roberta-large"
    model_type: str = "crf"
    dev_tokens_path: Optional[Path] = None
    checkpoint_path: Optional[Path] = None
    debug_output_path: Optional[Path] = None
    num_labels: int = 3
    batch_size: int = 8
    max_length: int = 256
    learning_rate: float = 5e-4
    weight_decay: float = 0.01
    momentum: float = 0.9
    optimizer_type: str = "adamw"
    epochs: int = 10
    validation_split: float = 0.1
    warmup_ratio: float = 0.1
    max_grad_norm: float = 1.0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    max_train_batches: Optional[int] = None
    max_eval_batches: Optional[int] = None
    dropout: float = 0.1
    aux_ce_weight: float = 0.3
    sentence_loss_weight: float = 0.5
    label_smoothing: float = 0.1
    early_stopping_patience: int = 2
    class_weights: Optional[Tuple[float, ...]] = None
    # Multi-granularity model specific knobs
    gate_function: str = "sigmoid"
    apply_gate_in_inference: bool = True
    use_crf: bool = True
    crf_loss_weight: float = 0.5
    enforce_bio_constraints: bool = True
    use_focal_loss: bool = True
    focal_alpha: float = 0.75
    focal_gamma: float = 3.0
    focal_warmup_steps: int = 3
    focal_warmup_method: str = "linear"

def train_one_epoch(
    model: torch.nn.Module,
    loader,
    optimizer,
    scheduler,
    device: torch.device,
    max_batches: Optional[int] = None,
    max_grad_norm: float = 1.0,
) -> float:
    """Run a single training epoch and return the average loss."""
    model.train()
    total_loss = 0.0
    steps = 0
    for step, batch in enumerate(loader):
        if max_batches is not None and step >= max_batches:
            break
        optimizer.zero_grad()
        batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
        # Filter inputs to match model signature unless it supports **kwargs
        fwd_sig = inspect.signature(model.forward)
        has_var_kw = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in fwd_sig.parameters.values())
        if has_var_kw:
            model_inputs = batch
        else:
            allowed = set(fwd_sig.parameters.keys())
            model_inputs = {k: v for k, v in batch.items() if k in allowed}
        outputs = model(**model_inputs)
        loss = outputs["loss"]
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item()
        steps += 1
    return total_loss / max(1, steps)

def evaluate_model(
    model: torch.nn.Module,
    loader,
    device: torch.device,
    tokenizer: AutoTokenizer,
    max_batches: Optional[int] = None,
    generate_debug_report: bool = False,
) -> Dict[str, Any]:
    """Compute loss and span-level precision/recall scores mirroring the official scorer."""
    model.eval()
    total_loss = 0.0
    steps = 0
    total_cumulative_prec = 0.0
    total_cumulative_rec = 0.0
    total_pred_span_count = 0
    total_gold_span_count = 0
    total_pred_malformed = 0
    total_gold_malformed = 0
    debug_report: List[str] = []
    print("\n--- Starting Validation ---")
    with torch.no_grad():
        for step, batch in enumerate(loader):
            if max_batches is not None and step >= max_batches:
                break
            batch_on_device = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
            fwd_sig = inspect.signature(model.forward)
            has_var_kw = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in fwd_sig.parameters.values())
            if has_var_kw:
                model_inputs = batch_on_device
            else:
                allowed = set(fwd_sig.parameters.keys())
                model_inputs = {k: v for k, v in batch_on_device.items() if k in allowed}
            outputs = model(**model_inputs)
            loss = outputs.get("loss")
            if loss is not None:
                total_loss += loss.item()
            labels = batch_on_device["labels"]
            mask = labels != -100
            predictions = outputs["predictions"]
            metrics_batch = calculate_span_metrics(predictions, labels, mask)
            total_cumulative_prec += metrics_batch["cumulative_prec"]
            total_cumulative_rec += metrics_batch["cumulative_rec"]
            total_pred_span_count += metrics_batch["pred_span_count"]
            total_gold_span_count += metrics_batch["gold_span_count"]
            total_pred_malformed += metrics_batch["pred_malformed"]
            total_gold_malformed += metrics_batch["gold_malformed"]
            if generate_debug_report:
                debug_report.extend(generate_evaluation_debug_report(step, batch_on_device, outputs, labels, mask, predictions, tokenizer))
            steps += 1
    avg_loss = total_loss / max(1, steps)
    # Calculate overall precision, recall, and F1
    precision = total_cumulative_prec / max(1, total_pred_span_count)
    recall = total_cumulative_rec / max(1, total_gold_span_count)
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    print(f"Validation complete. Loss: {avg_loss:.4f}, P: {precision:.4f}, R: {recall:.4f}, F1: {f1:.4f}")
    if total_pred_malformed > 0 or total_gold_malformed > 0:
        print(f"  Malformed predictions: {total_pred_malformed}, Malformed gold: {total_gold_malformed}")
    results = {
        "loss": avg_loss,
        "span_precision": precision,
        "span_recall": recall,
        "span_f1": f1,
    }
    if generate_debug_report:
        results["debug_report"] = debug_report
    return results

"""
DataLoader creation moved to process_data/dataloader.py (create_data_loaders).
"""

def _initialize_model(config: TrainConfig, device: torch.device) -> torch.nn.Module:
    """Instantiate the selected model type and move to device."""
    model_type = config.model_type.lower()
    if model_type in {"crf", "token_crf"}:
        model = TokenCRFModel(
            model_name=config.model_name,
            num_labels=config.num_labels,
            dropout=config.dropout,
            aux_ce_weight=config.aux_ce_weight,
            label_smoothing=config.label_smoothing,
            class_weights=config.class_weights,
        )
    elif model_type in {"multi_granularity", "multi-granularity"}:
        model = MultiGranularityTokenCRFModel(
            model_name=config.model_name,
            num_labels=config.num_labels,
            dropout=config.dropout,
            sentence_loss_weight=config.sentence_loss_weight,
            label_smoothing=config.label_smoothing,
            class_weights=config.class_weights,
            gate_function=config.gate_function,
            apply_gate_in_inference=config.apply_gate_in_inference,
            use_crf=config.use_crf,
            crf_loss_weight=config.crf_loss_weight,
            enforce_bio_constraints=config.enforce_bio_constraints,
            use_focal_loss=config.use_focal_loss,
            focal_alpha=config.focal_alpha,
            focal_gamma=config.focal_gamma,
            focal_warmup_steps=config.focal_warmup_steps,
            focal_warmup_method=config.focal_warmup_method,
        )
    elif model_type in {"baseline", "roberta_baseline", "roberta-baseline"}:
        model = RobertaLargeBaseline(
            model_name=config.model_name,
            num_labels=config.num_labels,
            dropout=config.dropout,
            label_smoothing=config.label_smoothing,
            class_weights=config.class_weights,
        )
    else:
        raise ValueError(f"Unknown model_type: {config.model_type}")

    model.to(device)
    return model

def _setup_optimizer_and_scheduler(model, config: TrainConfig, total_steps: int):
    if config.optimizer_type.lower() == "adamw":
        optimizer = prepare_adamw_optimizer(model, learning_rate=config.learning_rate, weight_decay=config.weight_decay)
    elif config.optimizer_type.lower() == "sgd":
        optimizer = prepare_optimizer(model, learning_rate=config.learning_rate, weight_decay=config.weight_decay, momentum=config.momentum)
    else:
        raise ValueError(f"Unsupported optimizer_type: {config.optimizer_type}")
    scheduler = create_scheduler(optimizer, total_steps=total_steps, warmup_ratio=config.warmup_ratio)
    return optimizer, scheduler

def _load_checkpoint(
    config: TrainConfig,
    model: torch.nn.Module,
    optimizer,
    scheduler,
    device: torch.device,
    start_epoch: int,
    best_val_f1: float,
    best_val_loss: float,
):
    if config.checkpoint_path and isinstance(config.checkpoint_path, Path) and config.checkpoint_path.exists():
        state = torch.load(config.checkpoint_path, map_location=device)
        model.load_state_dict(state.get("model_state_dict", {}))
        if "optimizer_state_dict" in state:
            optimizer.load_state_dict(state["optimizer_state_dict"])
        if "scheduler_state_dict" in state:
            scheduler.load_state_dict(state["scheduler_state_dict"])
        start_epoch = int(state.get("epoch", start_epoch)) + 1
        best_val_f1 = float(state.get("best_val_f1", best_val_f1))
        best_val_loss = float(state.get("best_val_loss", best_val_loss))
        print(f"Loaded checkpoint from {config.checkpoint_path}")
    return start_epoch, best_val_f1, best_val_loss

"""
Span metrics and debug-report helpers moved to training/utils.py.
"""

def train_model(
    config: TrainConfig,
    model_output_path: Path,
    *,
    train_loader=None,
    val_loader=None,
    tokenizer: Optional[AutoTokenizer] = None,
) -> None:
    """Orchestrate dataloading, training, and evaluation for the token model."""
    device = torch.device(config.device)
    # Initialization: tokenizer (for debug reporting) if not provided
    if tokenizer is None:
        tokenizer = load_tokenizer(config.model_name)
    # Require externally provided loaders
    if train_loader is None or val_loader is None:
        raise ValueError("train_loader and val_loader must be provided to train_model.")
    model = _initialize_model(config, device)
    total_steps = len(train_loader) * config.epochs
    optimizer, scheduler = _setup_optimizer_and_scheduler(model, config, total_steps)
    start_epoch = 1
    best_val_f1 = -1.0
    best_val_loss = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    start_epoch, best_val_f1, best_val_loss = _load_checkpoint(config, model, optimizer, scheduler, device, start_epoch, best_val_f1, best_val_loss)
    for epoch in range(start_epoch, config.epochs + 1):
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            device,
            max_batches=config.max_train_batches,
            max_grad_norm=config.max_grad_norm,
        )
        metrics = evaluate_model(
            model,
            val_loader,
            device,
            tokenizer=tokenizer,
            max_batches=config.max_eval_batches,
            generate_debug_report=False,
        )
        print(
            f"Epoch {epoch}/{config.epochs} - "
            f"train_loss: {train_loss:.4f} - "
            f"val_loss: {metrics['loss']:.4f} - "
            f"span_p: {metrics['span_precision']:.4f} - "
            f"span_r: {metrics['span_recall']:.4f} - "
            f"span_f1: {metrics['span_f1']:.4f}"
        )
        if metrics["span_f1"] > best_val_f1:
            best_val_f1 = metrics["span_f1"]
            best_state = {
                "epoch": epoch,
                "model_state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_f1": best_val_f1,
                "best_val_loss": best_val_loss,
            }
            print(f"New best F1 score: {best_val_f1:.4f}. Saving model state.")
            if model_output_path:
                torch.save(best_state, model_output_path)
            if config.debug_output_path and "debug_report" in metrics:
                with open(config.debug_output_path, "w") as f:
                    for line in metrics["debug_report"]:
                        f.write(line + "\n")
                print(f"Saved debug report to {config.debug_output_path}")
        if metrics["loss"] < best_val_loss:
            best_val_loss = metrics["loss"]
            epochs_without_improvement = 0
            best_epoch = epoch



def run_training(
    train_loader,
    val_loader,
    model_name: str,
    model_type: str = "crf",
    checkpoint_path: Optional[Path] = None,
    model_output_path: Optional[Path] = None,
    debug_output_path: Optional[Path] = None,
    # data & batching
    batch_size: int = 8,
    max_length: int = 256,
    num_labels: int = 3,
    # training schedule
    epochs: int = 60,
    learning_rate: float = 3e-5,
    optimizer_type: str = "adamw",
    weight_decay: float = 0.01,
    momentum: float = 0.9,
    warmup_ratio: float = 0.1,
    max_grad_norm: float = 1.0,
    max_train_batches: Optional[int] = None,
    max_eval_batches: Optional[int] = None,
    device: Optional[str] = None,
    # model regularization / losses
    dropout: float = 0.1,
    aux_ce_weight: float = 0.3,
    label_smoothing: float = 0.1,
    # multi-granularity model knobs
    sentence_loss_weight: float = 0.8,
    gate_function: str = "sigmoid",
    apply_gate_in_inference: bool = True,
    use_crf: bool = True,
    crf_loss_weight: float = 0.8,
    enforce_bio_constraints: bool = True,
    use_focal_loss: bool = True,
    focal_alpha: float = 0.35,
    focal_gamma: float = 2.0,
    focal_warmup_steps: int = 0,
    focal_warmup_method: str = "linear",
    # training controls
    early_stopping_patience: int = 2,
    class_weights: Optional[Tuple[float, ...]] = None,
    tokenizer: Optional[AutoTokenizer] = None,
) -> None:
    """Entry point to configure and execute token-level training in one call."""
    config = TrainConfig(
        checkpoint_path=checkpoint_path,
        debug_output_path=debug_output_path,
        model_name=model_name,
        model_type=model_type,
        num_labels=num_labels,
        batch_size=batch_size,
        max_length=max_length,
        epochs=epochs,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        momentum=momentum,
        optimizer_type=optimizer_type,
        warmup_ratio=warmup_ratio,
        max_grad_norm=max_grad_norm,
        device=device or ("cuda" if torch.cuda.is_available() else "cpu"),
        max_train_batches=max_train_batches,
        max_eval_batches=max_eval_batches,
        dropout=dropout,
        aux_ce_weight=aux_ce_weight,
        sentence_loss_weight=sentence_loss_weight,
        label_smoothing=label_smoothing,
        early_stopping_patience=early_stopping_patience,
        class_weights=class_weights,
        gate_function=gate_function,
        apply_gate_in_inference=apply_gate_in_inference,
        use_crf=use_crf,
        crf_loss_weight=crf_loss_weight,
        enforce_bio_constraints=enforce_bio_constraints,
        use_focal_loss=use_focal_loss,
        focal_alpha=focal_alpha,
        focal_gamma=focal_gamma,
        focal_warmup_steps=focal_warmup_steps,
        focal_warmup_method=focal_warmup_method,
    )
    train_model(
        config,
        model_output_path,
        train_loader=train_loader,
        val_loader=val_loader,
        tokenizer=tokenizer,
    )

# **Main pipeline**

In [12]:
# ----------------------------
# Configuration
# ----------------------------
ROOT_DIR = Path("/kaggle")
DATASETS_DIR = ROOT_DIR / "input" / "propaganda-dataset"/ "datasets_80_10_10"
TRAIN_ARTICLES_DIR = DATASETS_DIR / "train-articles"
TRAIN_SPANS_FILE = DATASETS_DIR / "train-task1-SI.labels"



DEV_ARTICLES_DIR = DATASETS_DIR / "dev-articles"
DEV_SPANS_FILE = DATASETS_DIR / "dev-task1-SI.labels"

TEST_ARTICLES_DIR = DATASETS_DIR / "test-articles"
TEST_SPAN_FILE = DATASETS_DIR / "test-task1-SI.labels"

TRAIN_OUTPUT_LABELED_TOKENS = ROOT_DIR / "working"/ "train-task1-tokens.tsv"
DEV_OUTPUT_LABELED_TOKENS = ROOT_DIR / "working"/"dev-task1-tokens.tsv"
TEST_OUTPUT_LABELED_TOKENS = ROOT_DIR / "working"/"test-task1-tokens.tsv"


DEBUG_OUTPUT_PATH = ROOT_DIR / "working" / "debug_report.txt"
DEFAULT_VOCAB_OUTPUT = ROOT_DIR / "working"/"train-task1-vocab.json"
TEST_OUTPUT_PRED= ROOT_DIR / "working"/"results.labels"

MODEL_OUTPUT_PATH= ROOT_DIR / "working" / "best_model.pt"

MODEL_NAME = "roberta-large"
MAX_TOKENS = 512
STRIDE = 128

MAX_SEQUENCE_LENGTH = 256
BATCH_SIZE = 8

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModel.from_pretrained(MODEL_NAME)
model.config.output_hidden_states = True
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 1024, padding_idx=1)
    (position_embeddings): Embedding(514, 1024, padding_idx=1)
    (token_type_embeddings): Embedding(1, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-23): 24 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      

1. Labeling data



In [13]:
# Process training data
create_token_labels(
    articles_dir=TRAIN_ARTICLES_DIR,
    spans_file=TRAIN_SPANS_FILE,
    tokens_output=TRAIN_OUTPUT_LABELED_TOKENS,
    tokenizer=tokenizer,
    model=model,
    device=device,
    max_tokens=MAX_TOKENS,
    stride=STRIDE
)

# Process development data
create_token_labels(
    articles_dir=DEV_ARTICLES_DIR,
    spans_file=DEV_SPANS_FILE,
    tokens_output=DEV_OUTPUT_LABELED_TOKENS,
    tokenizer=tokenizer,
    model=model,
    device=device,
    max_tokens=MAX_TOKENS,
    stride=STRIDE
)


/usr/local/lib/python3.11/dist-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.4). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


2. Create Vocab map

In [15]:
tokens = read_tokens(TRAIN_OUTPUT_LABELED_TOKENS, delimiter="\t", token_column=4)
vocab = build_vocab(
    tokens,
    tokenizer=tokenizer,
    special_tokens=[token for token in ["<pad>", "<unk>"]],
    min_frequency=1,
    max_size=None,
)
write_vocab(vocab, DEFAULT_VOCAB_OUTPUT)

3. Data Loader

In [16]:
vocab_map = load_vocab_mapping(DEFAULT_VOCAB_OUTPUT)
train_loader = create_data_loader(
    TRAIN_OUTPUT_LABELED_TOKENS,
    tokenizer,
    batch_size=BATCH_SIZE,
    shuffle=True,
    max_length=MAX_SEQUENCE_LENGTH,
    vocab_map=vocab_map,
)

dev_loader = create_data_loader(
    DEV_OUTPUT_LABELED_TOKENS,
    tokenizer,
    batch_size=BATCH_SIZE,
    shuffle=False,
    max_length=MAX_SEQUENCE_LENGTH,
    vocab_map=None,
)

# **4.Train Model**

In [ ]:
# ----------------------------
# Training
# ----------------------------
run_training(
  train_loader=train_loader,
  val_loader=dev_loader,
  model_name=MODEL_NAME,
  model_type="multi_granularity",
  debug_output_path=DEBUG_OUTPUT_PATH,
  model_output_path=MODEL_OUTPUT_PATH,
  num_labels=3,
  batch_size=BATCH_SIZE,
  max_length=MAX_SEQUENCE_LENGTH,
  tokenizer=tokenizer,
  sentence_loss_weight = 0.8,
  gate_function="sigmoid",
  apply_gate_in_inference = True,
  use_crf = True,
  crf_loss_weight = 0.8,
  enforce_bio_constraints = True,
  use_focal_loss = True,
  focal_alpha = 0.35,
  focal_gamma = 3.0,
  focal_warmup_steps = 3,
  focal_warmup_method = "linear",
  epochs=40
)

'''
run_training(
  train_loader=train_loader,
  val_loader=dev_loader,
  model_name=MODEL_NAME,
  model_type="roberta_baseline",
  debug_output_path=DEBUG_OUTPUT_PATH,
  class_weights=(0.38,27.63,2.83)
)
'''

Cleanup

In [ ]:
import gc
def _to_cpu_and_drop_grads(model: torch.nn.Module) -> None:
    """Move model to CPU and drop all grad tensors in-place."""
    try:
        model.eval()
    except Exception:
        pass
    try:
        model.to("cpu")
    except Exception:
        pass
    try:
        for p in model.parameters():
            p.grad = None
    except Exception:
        pass


def release_model(model: Optional[torch.nn.Module]) -> None:
    """Release a PyTorch model's GPU memory as much as possible.

    Notes:
    - This function operates in-place on the module (moving params to CPU and
      clearing grads). The caller should also drop all references to the model
      afterwards (e.g., `model = None`) to ensure full collection.
    """
    if model is None:
        return
    _to_cpu_and_drop_grads(model)


def release_optimizer(optimizer: Optional[torch.optim.Optimizer]) -> None:
    """Clear optimizer gradients and state to free RAM/VRAM.

    Warning: After this call, the optimizer should not be used again.
    """
    if optimizer is None:
        return
    try:
        optimizer.zero_grad(set_to_none=True)
    except Exception:
        pass
    # Clear moments/state
    try:
        optimizer.state.clear()
    except Exception:
        pass


def release_scheduler(scheduler) -> None:
    """Detach references held by scheduler.

    Most schedulers are lightweight, but we still clear common attributes.
    """
    if scheduler is None:
        return
    for attr in ("optimizer", "_optimizer"):
        if hasattr(scheduler, attr):
            try:
                setattr(scheduler, attr, None)
            except Exception:
                pass


def shutdown_dataloaders(dataloaders: Optional[Iterable]) -> None:
    """Attempt to shut down DataLoader workers to free RAM.

    Works best if loaders were created with persistent_workers=False.
    """
    if not dataloaders:
        return
    for loader in dataloaders:
        # Best-effort: trigger iterator shutdown if present
        try:
            it = getattr(loader, "_iterator", None)
            if it is not None and hasattr(it, "_shutdown_workers"):
                it._shutdown_workers()  # type: ignore[attr-defined]
        except Exception:
            pass


def empty_cuda_cache(sync: bool = True, reset_stats: bool = True) -> None:
    if torch.cuda.is_available():
        try:
            if sync:
                torch.cuda.synchronize()
        except Exception:
            pass
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
        if reset_stats:
            try:
                torch.cuda.reset_peak_memory_stats()
            except Exception:
                pass


def cleanup_after_training(
    *,
    model: Optional[torch.nn.Module] = None,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler=None,
    dataloaders: Optional[Iterable] = None,
    call_gc: bool = True,
    cuda_sync: bool = True,
    cuda_reset_stats: bool = True,
) -> None:
    """Centralized RAM/VRAM cleanup to call when training/evaluation is done.

    Example usage at end of training:
        cleanup_after_training(model=model, optimizer=optimizer, scheduler=scheduler, dataloaders=[train_loader, val_loader])
        model = None  # drop last reference in caller
    """
    release_model(model)
    release_optimizer(optimizer)
    release_scheduler(scheduler)
    shutdown_dataloaders(dataloaders)
    if call_gc:
        gc.collect()
    empty_cuda_cache(sync=cuda_sync, reset_stats=cuda_reset_stats)


def cleanup_after_inference(*, model: Optional[torch.nn.Module] = None, dataloaders: Optional[Iterable] = None) -> None:
    """Smaller variant for prediction-only scenarios."""
    release_model(model)
    shutdown_dataloaders(dataloaders)
    gc.collect()
    empty_cuda_cache()


**Scorer**

In [ ]:
Span = Tuple[int, int]
SpanMap = Dict[str, List[Span]]


def load_labels_file(path: Path | str) -> SpanMap:
    """Load Task SI .labels file (article_id<TAB>start<TAB>end per line)."""
    p = Path(path)
    mapping: SpanMap = {}
    if not p.exists():
        return mapping
    with p.open("r", encoding="utf8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 3:
                continue
            aid, s_str, e_str = parts[0], parts[1], parts[2]
            try:
                s, e = int(s_str), int(e_str)
            except ValueError:
                continue
            if e <= s:
                continue
            mapping.setdefault(aid, []).append((s, e))
    return mapping


def _overlap(a: Span, b: Span) -> int:
    s = max(a[0], b[0])
    e = min(a[1], b[1])
    return max(0, e - s)


def _best_overlap_length(span: Span, others: List[Span]) -> int:
    best = 0
    for o in others:
        ov = _overlap(span, o)
        if ov > best:
            best = ov
    return best


def compute_span_prf(pred: SpanMap, gold: SpanMap) -> Tuple[float, float, float]:
    """Compute span-level precision/recall/F1 using overlap-based scoring similar to the official scorer:
    - Precision: average over predicted spans of best_overlap_len / predicted_span_len
    - Recall:    average over gold spans of best_overlap_len / gold_span_len
    """
    cumulative_prec = 0.0
    cumulative_rec = 0.0
    pred_count = 0
    gold_count = 0

    article_ids = set(pred.keys()) | set(gold.keys())
    for aid in article_ids:
        p_spans = pred.get(aid, [])
        g_spans = gold.get(aid, [])

        for ps in p_spans:
            p_len = max(1, ps[1] - ps[0])
            bo = _best_overlap_length(ps, g_spans)
            cumulative_prec += bo / p_len
            pred_count += 1

        for gs in g_spans:
            g_len = max(1, gs[1] - gs[0])
            bo = _best_overlap_length(gs, p_spans)
            cumulative_rec += bo / g_len
            gold_count += 1

    precision = cumulative_prec / max(1, pred_count)
    recall = cumulative_rec / max(1, gold_count)
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return precision, recall, f1


# **Evaluate on test dataset**

In [ ]:
def _init_model(
    model_type: str,
    model_name: str,
    num_labels: int = 3,
    device: Optional[torch.device] = None,
) -> torch.nn.Module:
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    mt = model_type.lower()
    if mt in {"crf", "token_crf"}:
        model = TokenCRFModel(model_name=model_name, num_labels=num_labels)
    elif mt in {"multi_granularity", "multi-granularity"}:
        model = MultiGranularityTokenCRFModel(
            model_name=model_name,
            num_labels=num_labels,
            # keep defaults aligned with training
            use_crf=True,
            enforce_bio_constraints=True,
        )
    else:
        raise ValueError(f"Unsupported model_type: {model_type}")
    model.to(device)
    model.eval()
    return model


def _load_checkpoint_if_any(model: torch.nn.Module, checkpoint_path: Optional[Path], device: torch.device) -> None:
    if checkpoint_path is None:
        return
    ckpt = Path(checkpoint_path)
    if not ckpt.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt}")
    state = torch.load(ckpt, map_location=device)
    model.load_state_dict(state.get("model_state_dict", state))


def _batch_indices(total: int, batch_size: int) -> List[List[int]]:
    return [list(range(i, min(i + batch_size, total))) for i in range(0, total, batch_size)]


def _pred_labels_to_token_spans(pred_labels: List[int]) -> List[Tuple[int, int]]:
    """Convert BIO labels to (start_token_index, end_token_index) spans, end exclusive."""
    spans: List[Tuple[int, int]] = []
    start_idx: Optional[int] = None
    for i, tag in enumerate(pred_labels):
        if tag == 1:  # B
            if start_idx is not None and start_idx < i:
                spans.append((start_idx, i))
            start_idx = i
        elif tag == 2:  # I
            pass
        else:  # O
            if start_idx is not None and start_idx < i:
                spans.append((start_idx, i))
            start_idx = None
    if start_idx is not None and start_idx < len(pred_labels):
        spans.append((start_idx, len(pred_labels)))
    return spans


def _tokenspan_to_charspan(
    s_tok: int,
    e_tok: int,
    records: List[TokenRecord],
) -> Optional[Tuple[int, int]]:
    s_tok = max(0, min(s_tok, len(records) - 1))
    e_tok = max(s_tok + 1, min(e_tok, len(records)))
    start_span = parse_span(records[s_tok].span)
    end_span = parse_span(records[e_tok - 1].span)
    if not start_span or not end_span:
        return None
    start_char = start_span[0]
    end_char = end_span[1]
    if end_char <= start_char:
        return None
    return (start_char, end_char)


def _post_process_char_span(
    start_char: int,
    end_char: int,
    s_tok: int,
    e_tok: int,
    records: List[TokenRecord],
    tokenizer: AutoTokenizer,
    article_text: str,
) -> Tuple[int, int]:
    """Adjust span to better align with RoBERTa subword boundaries.
    - If the first token is a continuation subword (no 'Ġ' prefix), expand span backward by 1 char.
    - If the last character inside the span is not alphanumeric (e.g., whitespace/punct), shrink end by 1.
    Bounds are clamped and we ensure end > start.
    """
    try:
        first_token = records[s_tok].token
    except Exception:
        first_token = ""

    # RoBERTa BPE uses 'Ġ' to mark word starts; lack of it indicates a subword continuation
    if first_token and not first_token.startswith("Ġ") and start_char > 0:
        start_char = max(0, start_char - 1)

    # If end falls on a non-word char inside the span, pull it back by one
    if end_char > start_char:
        last_idx = end_char - 1
        if 0 <= last_idx < len(article_text):
            ch = article_text[last_idx]
            if not ch.isalnum():
                end_char = max(start_char + 1, end_char - 1)

    return (start_char, end_char)


def _merge_overlapping_spans(spans: List[Tuple[int, int]]) -> List[Tuple[int, int]]:
    """Merge overlapping or touching character spans.
    Assumes spans are (start, end) with end exclusive.
    """
    if not spans:
        return []
    spans_sorted = sorted(spans)
    merged: List[Tuple[int, int]] = []
    cur_s, cur_e = spans_sorted[0]
    for s, e in spans_sorted[1:]:
        if s <= cur_e:  # overlap or touch
            cur_e = max(cur_e, e)
        else:
            merged.append((cur_s, cur_e))
            cur_s, cur_e = s, e
    merged.append((cur_s, cur_e))
    return merged


def run_predictions(
    *,
    test_articles_dir: Path,
    test_gold_spans: Optional[Path],
    tokens_output: Path,
    predictions_output: Path,
    model_name: str = "roberta-large",
    model_type: str = "multi_granularity",
    checkpoint_path: Optional[Path] = None,
    batch_size: int = 8,
    max_length: int = 256,
) -> Dict[str, float]:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    # 1) Ensure tokens TSV exists for test (uses gold spans only to fill label column; model inference ignores them)
    create_token_labels(
        articles_dir=test_articles_dir,
        spans_file=test_gold_spans,
        tokens_output=tokens_output,
        tokenizer=tokenizer,
        model=None,  # not used inside create_token_labels
        device=device,
        max_tokens=512,
        stride=128,
    )

    # 2) Prepare chunks and dataset (keep the exact chunk list for offset mapping)
    rows_iter = read_token_rows(tokens_output)
    chunks: List[List[TokenRecord]] = list(group_token_records(rows_iter))
    dataset = PropagandaTokenDataset(chunks, tokenizer, max_length=max_length, vocab_map=None)
    collate_fn = build_collate_fn(getattr(tokenizer, "pad_token_id", 0))

    # 3) Initialize and load model
    model = _init_model(model_type, model_name, num_labels=3, device=device)
    _load_checkpoint_if_any(model, checkpoint_path, device)

    # 4) Inference in simple index-batches to preserve mapping to chunks
    has_cls = getattr(tokenizer, "cls_token_id", None) is not None
    has_sep = getattr(tokenizer, "sep_token_id", None) is not None
    special_count = (1 if has_cls else 0) + (1 if has_sep else 0)
    available_len = max_length - special_count

    predictions_by_article: Dict[str, List[Tuple[int, int]]] = defaultdict(list)
    article_text_cache: Dict[str, str] = {}

    for batch_indices in _batch_indices(len(dataset), batch_size):
        batch_items = [dataset[i] for i in batch_indices]
        batch = collate_fn(batch_items)
        batch_on_device = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
        with torch.no_grad():
            outputs = model(
                input_ids=batch_on_device["input_ids"],
                attention_mask=batch_on_device["attention_mask"],
                sentence_ids=batch_on_device.get("sentence_ids"),
            )
        preds = outputs["predictions"]  # List[List[int]] or tensor
        if isinstance(preds, torch.Tensor):
            preds = preds.cpu().tolist()

        for local_i, pred_seq in enumerate(preds):
            global_i = batch_indices[local_i]
            # Trim CLS/SEP predictions to align with records
            seq = list(map(int, pred_seq))
            if has_cls and len(seq) > 0:
                seq = seq[1:]
            if has_sep and len(seq) > 0:
                seq = seq[:-1]

            records = chunks[global_i]
            recs_trunc = records[:available_len] if available_len > 0 else []
            seq = seq[: len(recs_trunc)]
            if not seq or not recs_trunc:
                continue
            token_spans = _pred_labels_to_token_spans(seq)
            if not token_spans:
                continue
            article_id = recs_trunc[0].article_id
            # Load article text once
            if article_id not in article_text_cache:
                apath = resolve_article_path(article_id, test_articles_dir)
                article_text_cache[article_id] = load_article_text(apath)
            atext = article_text_cache[article_id]

            for s_tok, e_tok in token_spans:
                char_span = _tokenspan_to_charspan(s_tok, e_tok, recs_trunc)
                if not char_span:
                    continue
                s_char, e_char = char_span
                s_char, e_char = _post_process_char_span(s_char, e_char, s_tok, e_tok, recs_trunc, tokenizer, atext)
                if e_char > s_char:
                    predictions_by_article[article_id].append((s_char, e_char))

    # Optional: sort and deduplicate spans per article
    for aid in list(predictions_by_article.keys()):
        # Deduplicate and merge overlaps
        spans = list(set(predictions_by_article[aid]))
        predictions_by_article[aid] = _merge_overlapping_spans(spans)

    # 5) Write predictions in Task SI .labels format
    predictions_output.parent.mkdir(parents=True, exist_ok=True)
    with predictions_output.open("w", encoding="utf8") as out:
        def _aid_key(a: str) -> Tuple[int, str]:
            try:
                return (0, str(int(a)))
            except Exception:
                return (1, a)
        for aid in sorted(predictions_by_article.keys(), key=_aid_key):
            for s, e in sorted(predictions_by_article[aid]):
                out.write(f"{aid}\t{s}\t{e}\n")

    # 6) Evaluate using local scorer (logic aligned with official scorer)
    results = {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    if test_gold_spans and Path(test_gold_spans).exists():
        try:
            pred_map = load_labels_file(predictions_output)
            gold_map = load_labels_file(Path(test_gold_spans))
            p, r, f1 = compute_span_prf(pred_map, gold_map)
            results = {"precision": p, "recall": r, "f1": f1}
        except Exception as e:
            print(f"Warning: scorer evaluation failed: {e}")

    return results

In [ ]:
results = run_predictions(
    test_articles_dir=TEST_ARTICLES_DIR,
    test_gold_spans=TEST_SPAN_FILE, 
    tokens_output=TEST_OUTPUT_LABELED_TOKENS,
    predictions_output=TEST_OUTPUT_PRED,
    model_name="roberta-large",
    model_type="multi_granularity",
    checkpoint_path=MODEL_OUTPUT_PATH,
    batch_size=8,
    max_length=256,
)



In [34]:
import torch, gc

gc.collect()
torch.cuda.empty_cache()

